<a href="https://colab.research.google.com/github/malyalamounika051-bit/learning-github/blob/main/NotesGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, we need to install the required libraries: `langchain`, `langchain-google-genai`, and `gradio`.

In [1]:
pip install -U langchain-google-genai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0


To use the Gemini API, you'll need an API key. If you don't already have one, create a key in [Google AI Studio](https://makersuite.google.com/app/apikey).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`. Then, we'll retrieve it and configure the Generative AI library.

In [2]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Now, we'll set up the LangChain components. This includes initializing the Gemini model, creating a prompt template, and setting up a simple chain.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7, google_api_key=GOOGLE_API_KEY)

# Define the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant that generates comprehensive and well-structured notes on a given topic."),
    ("user", "Generate detailed notes on the following topic: {topic}")
])

# Create the LangChain chain
output_parser = StrOutputParser()
note_generation_chain = prompt | llm | output_parser

In [4]:
import gradio as gr

def generate_notes_gradio(topic):
    try:
        notes = note_generation_chain.invoke({"topic": topic})
        return notes
    except Exception as e:
        return f"An error occurred: {e}"

# Create the Gradio interface
iface = gr.Interface(
    fn=generate_notes_gradio,
    inputs=gr.Textbox(lines=2, placeholder="Enter a topic, e.g., 'The history of artificial intelligence'", label="Topic"),
    outputs=gr.Textbox(lines=15, label="Generated Notes", interactive=False),
    title="AI-Powered Notes Generator",
    description="Enter a topic, and I will generate comprehensive notes for you using Google Gemini and LangChain."
)

# Launch the Gradio app
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7aba817c10aa1b269.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
import google.generativeai as genai

for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025

Finally, we'll create a Gradio interface to make it easy to interact with our notes generator. You'll have an input box for the topic and an output box to display the generated notes.

In [6]:
import gradio as gr

def generate_notes_gradio(topic):
    try:
        notes = note_generation_chain.invoke({"topic": topic})
        return notes
    except Exception as e:
        return f"An error occurred: {e}"

# Create the Gradio interface
iface = gr.Interface(
    fn=generate_notes_gradio,
    inputs=gr.Textbox(lines=2, placeholder="Enter a topic, e.g., 'The history of artificial intelligence'", label="Topic"),
    outputs=gr.Textbox(lines=15, label="Generated Notes", interactive=False),
    title="AI-Powered Notes Generator (Gemini + LangChain)",
    description="Enter a topic, and I will generate comprehensive notes for you using Google Gemini and LangChain."
)

# Launch the Gradio app
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://946b71795d0178404d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
